# FedHealth — Notebook 04: CBC thalassemia risk-flag validation

## Purpose

FedHealth uses a routine CBC to flag **possible thalassemia risk**, alongside
Model A's healthy, anemia, and leukemia-risk outputs. This is an interpretable
screening layer, not a diagnosis and not a thalassemia subtype classifier.

It evaluates the existing CBC discriminant-index rule on `twoalphas.csv`.
Confirmation requires clinical review and haemoglobin electrophoresis.

In [ ]:
# --- Setup ------------------------------------------------------------------
# Module 3 Colab setup.
#
# Prerequisite:
# Clone the GitHub testing branch into /content/FedHealth-FYP before running
# these notebooks. Dataset CSVs remain in Google Drive at MyDrive/FYP.
#
# This repository stores shared files flat in "Module 3 ML/". The code below
# creates the temporary "fedhealth_ml" package layout expected by the notebooks.

from pathlib import Path
from google.colab import drive
import os
import shutil
import sys

REPO_DIR = Path('/content/FedHealth-FYP')
SOURCE_DIR = REPO_DIR / 'Module 3 ML'
assert SOURCE_DIR.exists(), (
    'Repository files are missing. Clone the GitHub testing branch first.'
)

if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

!pip -q install -r "/content/FedHealth-FYP/Module 3 ML/requirements.txt"

RUNTIME_DIR = Path('/content/fedhealth_runtime')
PACKAGE_DIR = RUNTIME_DIR / 'fedhealth_ml'
PACKAGE_DIR.mkdir(parents=True, exist_ok=True)

for filename in ['schema.py', 'preprocess.py', 'models.py', 'rules.py']:
    shutil.copy2(SOURCE_DIR / filename, PACKAGE_DIR / filename)

(PACKAGE_DIR / '__init__.py').write_text(
    '# Temporary Colab package for Module 3 validation.\n'
)

if str(RUNTIME_DIR) not in sys.path:
    sys.path.insert(0, str(RUNTIME_DIR))

DATA_DIR = '/content/drive/MyDrive/FYP'
ANEMIA = f'{DATA_DIR}/anemia_diagnosed_cbc_data_v4.csv'
THAL   = f'{DATA_DIR}/thalassemia_alphanorm.csv'
LEUK   = f'{DATA_DIR}/biased_leukemia_dataset.csv'

for dataset_path in [ANEMIA, THAL, LEUK]:
    assert os.path.exists(dataset_path), f'Dataset not found: {dataset_path}'

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from fedhealth_ml import schema, preprocess as pp

print('fedhealth_ml loaded. Model A features:', len(schema.MODEL_A_FEATURES))
print('All three datasets found in Google Drive.')


## 1. Load the validation dataset

`twoalphas.csv` contains known alpha-trait and silent-carrier records. It has no
healthy/control group, so it can evaluate rule behaviour on known
thalassemia-related records only; it cannot measure specificity or false-positive rate.

In [ ]:
from pathlib import Path
from fedhealth_ml import rules

THAL_CSV = f"{DATA_DIR}/twoalphas.csv"
assert Path(THAL_CSV).exists(), f"Dataset not found: {THAL_CSV}"

raw = pd.read_csv(THAL_CSV)
required = {"phenotype", "hb", "rbc", "mcv", "mch"}
missing = required - set(raw.columns)
assert not missing, f"Missing required columns: {sorted(missing)}"

print("Dataset:", THAL_CSV)
print("Records:", len(raw))
print("Phenotypes:", raw["phenotype"].value_counts().to_dict())

## 2. Apply the CBC risk flag

The clinical gate makes the rule applicable only to microcytic CBC patterns.
A positive flag means that a consensus of available indices favours a
thalassemia-trait pattern over iron deficiency. It is not a confirmed diagnosis.

In [ ]:
records = []

for _, row in raw.iterrows():
    cbc = {
        key: (None if pd.isna(value) else value)
        for key, value in row.to_dict().items()
    }
    result = rules.thalassemia_trait_flag(cbc)

    records.append({
        "phenotype": cbc["phenotype"],
        "rule_applicable": bool(result["applicable"]),
        "risk_flagged": bool(result["flag"]),
        "consensus": float(result["consensus"]),
        "n_indices": int(result["n_indices"]),
        "mentzer_index": result["mentzer_index"],
        "rationale": result["rationale"],
    })

results = pd.DataFrame(records)

summary = results.groupby("phenotype", dropna=False).agg(
    records=("phenotype", "size"),
    rule_applicable=("rule_applicable", "sum"),
    risk_flagged=("risk_flagged", "sum"),
    mean_consensus=("consensus", "mean"),
)
summary["applicable_rate"] = summary["rule_applicable"] / summary["records"]
summary["flag_rate"] = summary["risk_flagged"] / summary["records"]

print("THALASSEMIA RISK-FLAG VALIDATION")
print(summary.round(3))

## 3. Strict verification and evidence export

The CSV outputs are saved to Drive as reproducible validation evidence. No model
weights are trained or saved because this notebook validates a transparent rule,
rather than a subtype classifier.

In [ ]:
OUT = Path("/content/drive/MyDrive/FYP/validation/module3/thal_risk")
OUT.mkdir(parents=True, exist_ok=True)

results.to_csv(OUT / "twoalphas_rule_results.csv", index=False)
summary.to_csv(OUT / "twoalphas_rule_summary.csv")

assert len(results) == len(raw), "Every input record must receive a rule result."
assert results["rule_applicable"].dtype == bool
assert results["risk_flagged"].dtype == bool
assert results["consensus"].between(0, 1).all()
assert set(summary.index) == set(raw["phenotype"].dropna().unique())

print("NOTEBOOK 04 VALIDATION: PASS")
print("Saved results to:", OUT)
print("Limitation: no healthy/control group; this does not measure specificity.")

## Conclusion

Notebook 04 provides a transparent CBC-based thalassemia **risk flag** for the
system's screening workflow. It does not claim to diagnose thalassemia, infer a
subtype, or replace confirmatory testing.